In [1]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# 1. Generate Historical Stock Price Dataset (1 Year of Trading Days)
np.random.seed(42)
dates = pd.bdate_range(start="2025-01-01", periods=252)

daily_returns = np.random.normal(loc=0.0008, scale=0.015, size=len(dates))
shock_indices = [45, 110, 175, 220]
daily_returns[shock_indices] = [0.065, -0.058, 0.072, -0.061]

price_series = 150 * np.cumprod(1 + daily_returns)
df = pd.DataFrame({
    "Date": dates,
    "Close": np.round(price_series, 2),
})
df["Open"] = np.round(df["Close"].shift(1).fillna(150.0), 2)
df["High"] = np.round(
    df[["Open", "Close"]].max(axis=1)
    * (1 + np.abs(np.random.normal(0, 0.008, len(df)))),
    2,
)
df["Low"] = np.round(
    df[["Open", "Close"]].min(axis=1)
    * (1 - np.abs(np.random.normal(0, 0.008, len(df)))),
    2,
)
df["Volume"] = np.random.randint(1_500_000, 5_000_000, size=len(df))

# 2. Feature Engineering & Anomaly Detection
df["Daily_Return_Pct"] = df["Close"].pct_change().fillna(0) * 100
df["SMA_20"] = df["Close"].rolling(window=20, min_periods=1).mean().round(2)
df["SMA_50"] = df["Close"].rolling(window=50, min_periods=1).mean().round(2)
df["Rolling_Volatility"] = (
    df["Daily_Return_Pct"].rolling(window=20, min_periods=1).std().fillna(0)
)

mean_ret = df["Daily_Return_Pct"].mean()
std_ret = df["Daily_Return_Pct"].std()
df["Z_Score"] = (df["Daily_Return_Pct"] - mean_ret) / std_ret
df["Is_Anomaly"] = df["Z_Score"].abs() > 2.5

df.to_csv("stock_data_cleaned.csv", index=False)

# 3. Chart 1: Stock Price & Moving Averages
plt.figure(figsize=(11, 5))
plt.plot(
    df["Date"], df["Close"], label="Close Price ($)", color="#1f77b4", alpha=0.8
)
plt.plot(
    df["Date"],
    df["SMA_20"],
    label="20-Day SMA",
    color="#ff7f0e",
    linestyle="--",
)
plt.plot(
    df["Date"],
    df["SMA_50"],
    label="50-Day SMA",
    color="#2ca02c",
    linestyle="-.",
)
plt.title("Stock Price Trend with 20-Day and 50-Day Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.legend()
plt.tight_layout()
plt.savefig("stock_price_moving_averages.png", dpi=300)
plt.close()

# 4. Chart 2: Anomaly Detection Chart
plt.figure(figsize=(11, 5))
plt.plot(
    df["Date"],
    df["Close"],
    label="Close Price ($)",
    color="#4b5563",
    alpha=0.75,
)
anomalies = df[df["Is_Anomaly"]]
plt.scatter(
    anomalies["Date"],
    anomalies["Close"],
    color="#dc2626",
    s=70,
    zorder=5,
    label="Detected Anomaly (|Z| > 2.5)",
)
plt.title("Stock Price Anomaly Detection (Statistical Z-Score Method)")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.legend()
plt.tight_layout()
plt.savefig("stock_price_anomalies.png", dpi=300)
plt.close()

# 5. Chart 3: Daily Returns Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df["Daily_Return_Pct"], bins=30, kde=True, color="#2563eb")
plt.axvline(
    mean_ret + 2.5 * std_ret,
    color="#dc2626",
    linestyle="--",
    label="Anomaly Threshold (+2.5σ)",
)
plt.axvline(
    mean_ret - 2.5 * std_ret,
    color="#dc2626",
    linestyle="--",
    label="Anomaly Threshold (-2.5σ)",
)
plt.title("Distribution of Daily Percentage Returns")
plt.xlabel("Daily Return (%)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig("daily_returns_distribution.png", dpi=300)
plt.close()

print(
    f"Success! Cleaned dataset and all 3 PNG charts saved! Anomalies detected: {len(anomalies)}"
)

Success! Cleaned dataset and all 3 PNG charts saved! Anomalies detected: 6


### Stock Price Visualization & Anomaly Report

#### Three Actionable Observations:
1. **Moving Average Crossovers:** The 20-day Simple Moving Average (`SMA_20`) tracks short-term momentum closely, while crossovers above the 50-day Moving Average (`SMA_50`) signal sustained upward trend phases.
2. **Statistical Return Anomalies:** Using a Z-score threshold of $\vert{}Z\vert{} > 2.5$, extreme single-day price shocks (>5% daily swings) are isolated cleanly from normal daily market noise.
3. **Return Distribution & Volatility:** Daily percentage returns follow a near-normal distribution centered slightly above 0%, with tail outliers representing high-volatility breakout events.

#### Conclusion
Combining rolling moving averages (`SMA_20` and `SMA_50`) with statistical Z-score thresholding provides both trend visibility and automated anomaly detection. Analysts can use these signals to separate routine price fluctuations from significant market shocks.